# VQE Benchmark Framework
### Modular, reproducible VQE benchmark for molecular quantum chemistry

**Targets:** Qiskit 2.3.0 · Qiskit Nature 0.7+ · Qiskit Algorithms 0.3+  
**Molecules:** H₂, LiH, H₂O, CH₄, N₂, CO  
**Orbital spaces:** Full · Freeze Core · Active Space · FC + AS  
**Ansatze:** UCCSD · UCCGSD · k-UpCCGSD · HEA · ADAPT-VQE  

---
**Workflow cells:**  
1 Experiment Setup → 2 Molecular Problem → 3 Freeze Core (opt.) → 4 Active Space (opt.)  
→ 5 Fermionic Hamiltonian → 6 Qubit Mapping → 7 Build VQE → 8 Run VQE → 9 Analysis


## Cell 1 — Experiment Setup
Define all hyperparameters in one place. Change only this cell to switch molecules, mappers, ansatze, optimizers, and simulation modes.

In [1]:
# ============================================================
# CELL 1 — EXPERIMENT SETUP
# ============================================================

import sys
import os
import numpy as np
from pathlib import Path

# ── Resolve project paths ─────────────────────────────────
# HPC-safe: set PROJECT_ROOT eksplisit, tidak bergantung pada cwd
PROJECT_ROOT = Path("/home/mkhairiansyah/Project")   # ← ganti dengan path absolut kamu

SRC_DIR     = PROJECT_ROOT / "src"
CONFIGS_DIR = PROJECT_ROOT / "configs"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = PROJECT_ROOT/"tables"
# Verify
assert SRC_DIR.exists(),     f"SRC_DIR tidak ditemukan: {SRC_DIR}"
assert CONFIGS_DIR.exists(), f"CONFIGS_DIR tidak ditemukan: {CONFIGS_DIR}"
assert TABLES_DIR.exists(), f"TABLES_DIR tidak ditemukan: {TABLES_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for _p in [SRC_DIR, CONFIGS_DIR,TABLES_DIR]:
    _ps = str(_p)
    if _ps not in sys.path:
        sys.path.insert(0, _ps)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SRC_DIR      : {SRC_DIR}  (exists: {SRC_DIR.exists()})")

# ── Experiment Configuration ──────────────────────────────
EXPERIMENT = {
    # Molecule
    "molecule":         "CH4",       # H2 | LiH | H2O | CH4 | N2 | CO
    "basis":            "sto-3g",
    "charge":           0,
    "multiplicity":     1,          # 2S+1; spin = multiplicity - 1

    # Orbital space
    "freeze_core":      True,  # True  → apply FreezeCoreTransformer
    "use_active_space": True,  # True  → apply ActiveSpaceTransformer
    "active_electrons": 8,      # e.g. 4  (set when use_active_space=True)
    "active_orbitals":  8,      # e.g. 4  (set when use_active_space=True)

    # Qubit mapping
    "mapper":           "bravyi-kitaev",   # jordan-wigner | parity | bravyi-kitaev

    # Ansatz  (k-upccgsd recommended for PES scan — faster than adapt-vqe)
    "ansatz":           "adapt_vqe",# uccsd | uccgsd | k-upccgsd | hea | adapt-vqe
    "reps":             1,          # UCCSD/UCCGSD/HEA repetition layers
    "k":                2,          # k-UpCCGSD layers

    # Optimizer
    "optimizer":        "l_bfgs_b",    # cobyla | slsqp | l_bfgs_b | spsa
    "maxiter":          500,        
    "tol":              1e-6,

    # Initial point
    "initial_point":    "mp2",       # hf | mp2 | zero | random

    # Estimator / simulation
    "noise":            False,      # False → StatevectorEstimator (exact)
    "shots":            2048,       # Shots for noisy simulation
    "seed":             123,        # RNG seed for reproducibility

    # ── PES Scan ─────────────────────────────────────────
    "scan_type":    "single_ch",  # 'single_ch' (asymmetric) | 'symmetric'
    "r_min":        0.50,         # Å — start of scan range 0.50, 2.50, 21
    "r_max":        2.50,         # Å — end of scan range
    "r_npoints":    21,           # number of equidistant scan points
    "warm_start":   True,         # reuse previous θ* as initial point
}

# ── Reproducibility ───────────────────────────────────────
np.random.seed(EXPERIMENT["seed"])

# ── Print configuration summary ───────────────────────────
print("=" * 58)
print("   VQE PES SCAN — EXPERIMENT CONFIGURATION")
print("=" * 58)
for _k, _v in EXPERIMENT.items():
    print(f"  {_k:<22}: {_v}")
print("=" * 58)
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Results dir  : {RESULTS_DIR}")
print("=" * 58)


PROJECT_ROOT : /home/mkhairiansyah/Project
SRC_DIR      : /home/mkhairiansyah/Project/src  (exists: True)
   VQE PES SCAN — EXPERIMENT CONFIGURATION
  molecule              : CH4
  basis                 : sto-3g
  charge                : 0
  multiplicity          : 1
  freeze_core           : True
  use_active_space      : True
  active_electrons      : 8
  active_orbitals       : 8
  mapper                : bravyi-kitaev
  ansatz                : adapt_vqe
  reps                  : 1
  k                     : 2
  optimizer             : l_bfgs_b
  maxiter               : 500
  tol                   : 1e-06
  initial_point         : mp2
  noise                 : False
  shots                 : 2048
  seed                  : 123
  scan_type             : single_ch
  r_min                 : 0.5
  r_max                 : 2.5
  r_npoints             : 21
  warm_start            : True
  Project root : /home/mkhairiansyah/Project
  Results dir  : /home/mkhairiansyah/Project/results


## Cell 2 — Geometry Builder

Defines `build_geometry(molecule, r_scan, scan_type)` for parametric PES geometries.
For CH₄:
- `scan_type='single_ch'` — stretches **one** C-H bond only (asymmetric stretch, matches ORCA reference)
- `scan_type='symmetric'` — scales **all four** C-H bonds uniformly

For H₂, LiH, N₂, CO: `r_scan` sets the primary bond length.  
For H₂O: `r_scan` sets the symmetric O-H bond length (bond angle fixed).


In [2]:
# ============================================================
# CELL 2 — GEOMETRY BUILDER
# ============================================================
# Provides build_geometry(molecule, r_scan, scan_type) used by
# the PES scan loop (Cell 3) to generate atom strings at each
# scan coordinate without rewriting geometry logic each time.
# ============================================================

import math
import numpy as np

# ── Equilibrium bond lengths (Angstrom) ───────────────────
R_EQ = {
    "H2":  0.735,
    "LIH": 1.595,
    "H2O": 0.9572,
    "CH4": 1.0893,   # sqrt(3) * 0.629 Å
    "N2":  1.098,
    "CO":  1.128,
}


def build_geometry(
    molecule: str,
    r_scan: float | None = None,
    scan_type: str = "single_ch",
) -> str:
    """
    Build a PySCFDriver-compatible atom string at bond length r_scan.

    Parameters
    ----------
    molecule  : molecule key (H2, LiH, H2O, CH4, N2, CO)
    r_scan    : scan coordinate in Angstrom; None → equilibrium geometry
    scan_type : CH4 only — 'single_ch' | 'symmetric'
    """
    mol_key = molecule.upper()
    r_eq    = R_EQ.get(mol_key, 1.0)
    r       = r_eq if r_scan is None else float(r_scan)

    if mol_key == "H2":
        return f"H 0.0 0.0 0.0; H 0.0 0.0 {r:.6f}"

    elif mol_key == "LIH":
        return f"Li 0.0 0.0 0.0; H 0.0 0.0 {r:.6f}"

    elif mol_key == "H2O":
        # Symmetric O-H stretch; H-O-H angle fixed at 104.52°
        half = math.radians(104.52 / 2.0)
        hx   = r * math.sin(half)
        hy   = r * math.cos(half)
        return (
            f"O  0.000000  0.000000 0.0; "
            f"H  {hx:.6f}  {hy:.6f} 0.0; "
            f"H -{hx:.6f}  {hy:.6f} 0.0"
        )

    elif mol_key == "CH4":
        # Four tetrahedral unit vectors
        tet = np.array([
            [ 1,  1,  1],
            [-1, -1,  1],
            [-1,  1, -1],
            [ 1, -1, -1],
        ], dtype=float)
        tet /= np.linalg.norm(tet[0])   # → unit vectors

        if scan_type == "symmetric":
            # Scale ALL C-H bonds to r
            h_pos = tet * r
        else:  # "single_ch" — only H1 stretched; H2-H4 at equilibrium
            h_pos      = tet * r_eq
            h_pos[0]   = tet[0] * r

        lines = ["C  0.000000  0.000000  0.000000"]
        for x, y, z in h_pos:
            lines.append(f"H  {x:.6f}  {y:.6f}  {z:.6f}")
        return "; ".join(lines)

    elif mol_key == "N2":
        return f"N 0.0 0.0 0.0; N 0.0 0.0 {r:.6f}"

    elif mol_key == "CO":
        return f"C 0.0 0.0 0.0; O 0.0 0.0 {r:.6f}"

    else:
        raise ValueError(
            f"Molecule '{molecule}' not supported. "
            f"Supported: H2, LiH, H2O, CH4, N2, CO"
        )


# ── Verify equilibrium geometry ───────────────────────────
_mol    = EXPERIMENT["molecule"]
_stype  = EXPERIMENT["scan_type"]
_r_eq   = R_EQ.get(_mol.upper(), 1.0)

print(f"Geometry builder ready for : {_mol}")
print(f"Equilibrium bond length    : {_r_eq:.4f} Å")
print(f"Scan type                  : {_stype}")
print()
_test = build_geometry(_mol, _r_eq, _stype)
print("Equilibrium geometry:")
for _line in _test.split("; "):
    print(f"  {_line}")
print()
r_scan_grid = np.linspace(EXPERIMENT["r_min"], EXPERIMENT["r_max"], EXPERIMENT["r_npoints"])
print(f"Scan grid  : {r_scan_grid[0]:.3f} … {r_scan_grid[-1]:.3f} Å  "
      f"({len(r_scan_grid)} points, step {r_scan_grid[1]-r_scan_grid[0]:.4f} Å)")


Geometry builder ready for : CH4
Equilibrium bond length    : 1.0893 Å
Scan type                  : single_ch

Equilibrium geometry:
  C  0.000000  0.000000  0.000000
  H  0.628908  0.628908  0.628908
  H  -0.628908  -0.628908  0.628908
  H  -0.628908  0.628908  -0.628908
  H  0.628908  -0.628908  -0.628908

Scan grid  : 0.500 … 2.500 Å  (21 points, step 0.1000 Å)


## Cell 3 — PES Scan Main Loop

Iterates over the bond-length grid defined in Cell 1 (`r_min`→`r_max`, `r_npoints` steps).
At each geometry:
1. Rebuilds the `ElectronicStructureProblem` via PySCFDriver
2. Applies FreezeCoreTransformer and/or ActiveSpaceTransformer (if enabled)
3. Generates the qubit Hamiltonian (Fermionic → Pauli via the configured mapper)
4. Computes the FCI reference with NumPy exact diagonalisation
5. Runs VQE (k-UpCCGSD or any other configured ansatz)
6. Stores per-point results and writes a checkpoint file after every point

**Warm-start** (`warm_start=True`): optimised parameters θ* from point *i* are
used as the initial point for point *i+1*, significantly reducing the number
of VQE iterations near equilibrium where the PES is smooth.

**Checkpoint**: results are appended to `pes_*_checkpoint.json` after each
geometry, so the scan can be resumed if interrupted.


In [ ]:
# ============================================================
# CELL 3 — PES SCAN MAIN LOOP
# ============================================================

import time
import json
import numpy as np

from qiskit_nature.second_q.drivers       import PySCFDriver
from qiskit_nature.units                  import DistanceUnit
from qiskit_nature.second_q.transformers  import FreezeCoreTransformer, ActiveSpaceTransformer
from qiskit.quantum_info                  import SparsePauliOp
from qiskit_algorithms.minimum_eigensolvers import (
    VQE as VQEAlgorithm, NumPyMinimumEigensolver,
)

from mapper_factory         import get_mapper
from ansatz_factory         import get_ansatz
from adapt_vqe_factory      import build_adapt_vqe
from optimizer_factory      import get_optimizer
from initial_point_factory  import get_initial_point
from estimator_factory      import get_estimator

# ── Scan grid & state ─────────────────────────────────────
_ADAPT_KEYS = {"adapt-vqe", "adapt_vqe", "adaptvqe", "adapt"}
_is_adapt   = EXPERIMENT["ansatz"].lower() in _ADAPT_KEYS

r_scan_grid = np.linspace(
    EXPERIMENT["r_min"], EXPERIMENT["r_max"], EXPERIMENT["r_npoints"]
)

pes_results: list[dict] = []
prev_params = None          # warm-start carrier

# ── Checkpoint ────────────────────────────────────────────
_tag = (
    f"{EXPERIMENT['molecule']}_"
    f"{EXPERIMENT['basis']}_"
    f"{EXPERIMENT['ansatz']}_"
    f"{EXPERIMENT['mapper']}_"
    f"{EXPERIMENT['optimizer']}_"
    f"{EXPERIMENT['scan_type']}"
)
CHECKPOINT_FILE = RESULTS_DIR / f"pes_{_tag}_checkpoint.json"

if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE) as _f:
        _ckpt = json.load(_f)
    pes_results   = _ckpt.get("pes_results", [])
    _done_r       = {round(p["r"], 6) for p in pes_results}
    print(f"Resuming from checkpoint — {len(pes_results)} points already done.")
else:
    _done_r = set()

print("=" * 58)
print(f"  PES SCAN  —  {EXPERIMENT['molecule']} / "
      f"{EXPERIMENT['ansatz'].upper()} / {EXPERIMENT['basis']}")
print(f"  r = {r_scan_grid[0]:.3f} … {r_scan_grid[-1]:.3f} Å  "
      f"({len(r_scan_grid)} points)")
print(f"  scan_type : {EXPERIMENT['scan_type']}  "
      f"| warm_start : {EXPERIMENT['warm_start']}")
print("=" * 58)

# ── Main loop ─────────────────────────────────────────────
for _idx, _r in enumerate(r_scan_grid):
    _r_key = round(float(_r), 6)
    if _r_key in _done_r:
        print(f"  [skip]  r = {_r:.4f} Å  (checkpoint)")
        continue

    print(f"\n{'─'*58}")
    print(f"  Point {_idx+1}/{len(r_scan_grid)}  |  r = {_r:.4f} Å")
    print(f"{'─'*58}")

    # 1. Geometry
    _atom = build_geometry(
        EXPERIMENT["molecule"], _r, EXPERIMENT["scan_type"]
    )

    # 2. Molecular problem
    _spin   = EXPERIMENT["multiplicity"] - 1
    _driver = PySCFDriver(
        atom=_atom, unit=DistanceUnit.ANGSTROM,
        charge=EXPERIMENT["charge"], spin=_spin,
        basis=EXPERIMENT["basis"],
    )
    _prob_raw = _driver.run()   # simpan raw untuk MP2 initial point
    _prob     = _prob_raw

    # 3. Freeze core
    if EXPERIMENT["freeze_core"]:
        _prob = FreezeCoreTransformer().transform(_prob)

    # 4. Active space
    if EXPERIMENT["use_active_space"]:
        _prob = ActiveSpaceTransformer(
            num_electrons=EXPERIMENT["active_electrons"],
            num_spatial_orbitals=EXPERIMENT["active_orbitals"],
        ).transform(_prob)

    # 5. Fermionic Hamiltonian
    _sqop = _prob.hamiltonian.second_q_op()

    # 6. Qubit mapping + constant energy terms
    _use_2qr  = (EXPERIMENT["mapper"] == "parity")
    _mapper   = get_mapper(
        name=EXPERIMENT["mapper"],
        num_particles=_prob.num_particles if _use_2qr else None,
    )
    _qham     = _mapper.map(_sqop)
    _const    = sum(_prob.hamiltonian.constants.values())
    _id_op    = SparsePauliOp(['I' * _qham.num_qubits], coeffs=[_const])
    _qham     = (_qham + _id_op).simplify()

    # 7. FCI reference (NumPy exact diagonalisation)
    _fci_res  = NumPyMinimumEigensolver().compute_minimum_eigenvalue(_qham)
    _fci_e    = float(_fci_res.eigenvalue.real)

    # 8. VQE components
    _estimator = get_estimator(
        noisy=EXPERIMENT["noise"],
        shots=EXPERIMENT["shots"], seed=EXPERIMENT["seed"],
    )
    _optimizer = get_optimizer(
        name=EXPERIMENT["optimizer"],
        maxiter=EXPERIMENT["maxiter"], tol=EXPERIMENT["tol"],
    )

    _eh: list[float] = []          # energy history for this point

    def _cb(ec, params, mean, meta):
        _eh.append(float(mean))
        if len(_eh) % 50 == 0:
            print(f"    iter {len(_eh):4d} | E = {mean:.8f} Ha")

    # 9. Run VQE
    _t0 = time.perf_counter()

    if _is_adapt:
        _adapt, _pool = build_adapt_vqe(
            estimator=_estimator, optimizer=_optimizer,
            problem=_prob, mapper=_mapper,
            threshold=1e-5, max_iterations=20,
        )
        _adapt.solver.callback = _cb
        _res = _adapt.compute_minimum_eigenvalue(_qham)
        prev_params = None   # ADAPT manages params internally

    else:
        _ansatz = get_ansatz(
            name=EXPERIMENT["ansatz"], problem=_prob,
            mapper=_mapper, reps=EXPERIMENT["reps"], k=EXPERIMENT["k"],
        )
        # ── TAMBAHAN: dekomposisi PauliEvolutionGate ─────────────
        # Wajib untuk ansatz berbasis EvolvedOperatorAnsatz (PUCCD, UCCSD, UCCGSD)
        # agar StatevectorEstimator tidak mencoba materialisasi matriks 2^n × 2^n
        from qiskit.compiler import transpile as _transpile

        _ansatz = _transpile(
            _ansatz,
            basis_gates=["cx", "u", "rz", "ry", "rx", "x", "h"],
            optimization_level=1,
        )
        # ─────────────────────────────────────────────────────────
        # Warm-start: use previous θ* if parameter count matches
        # Adaptive Opsi C:
        # error < 10 mHa → warm-start
        # error >= 10 mHa atau titik pertama → MP2 reset
        _prev_err = pes_results[-1]["error_mha"] if pes_results else None

        if EXPERIMENT["warm_start"] and prev_params is not None and \
                len(prev_params) == _ansatz.num_parameters and \
                (_prev_err is None or _prev_err < 10.0):
            _init = np.array(prev_params)
            _init_strategy = "warm-start"
        else:
            _init = get_initial_point(
                name=EXPERIMENT["initial_point"], ansatz=_ansatz,
                problem=_prob_raw,      # raw problem untuk MP2
                seed=EXPERIMENT["seed"],
            )
            _init_strategy = f"MP2 reset (prev_err={_prev_err:.3f} mHa)" \
                             if _prev_err is not None else "MP2 (titik pertama)"

        print(f"  Initial point  : {_init_strategy}")

        _vqe = VQEAlgorithm(
            estimator=_estimator, ansatz=_ansatz,
            optimizer=_optimizer, initial_point=_init,
            callback=_cb,
        )
        _res = _vqe.compute_minimum_eigenvalue(_qham)

        # Save optimized params for next warm-start
        if hasattr(_res, "optimal_point") and _res.optimal_point is not None:
            prev_params = list(_res.optimal_point)
        elif hasattr(_res, "optimal_parameters") and _res.optimal_parameters is not None:
            prev_params = [_res.optimal_parameters[p] for p in _ansatz.parameters]

    _t1      = time.perf_counter()
    _vqe_e   = float(_res.eigenvalue.real)
    _err_mha = abs(_vqe_e - _fci_e) * 1000.0
    _nevals  = (
        _res.cost_function_evals
        if hasattr(_res, "cost_function_evals") and _res.cost_function_evals
        else len(_eh)
    )

    print(f"  VQE : {_vqe_e:.8f} Ha")
    print(f"  FCI : {_fci_e:.8f} Ha")
    print(f"  Err : {_err_mha:.4f} mHa  {'✓' if _err_mha < 1.0 else '✗'}")
    print(f"  Evals {_nevals}  |  Runtime {_t1-_t0:.1f} s")

    pes_results.append({
        "r":          float(_r),
        "vqe_energy": _vqe_e,
        "fci_energy": _fci_e,
        "error_mha":  _err_mha,
        "runtime":    round(_t1 - _t0, 3),
        "n_evals":    int(_nevals),
    })

    # Checkpoint after each point
    with open(CHECKPOINT_FILE, "w") as _f:
        json.dump(
            {"pes_results": pes_results, "experiment": {k: str(v) for k, v in EXPERIMENT.items()}},
            _f, indent=2,
        )

print()
print("=" * 58)
print(f"  Scan complete — {len(pes_results)} / {len(r_scan_grid)} points")
print("=" * 58)


  PES SCAN  —  CH4 / ADAPT_VQE / sto-3g
  r = 0.500 … 2.500 Å  (21 points)
  scan_type : single_ch  | warm_start : True

──────────────────────────────────────────────────────────
  Point 1/21  |  r = 0.5000 Å
──────────────────────────────────────────────────────────


/mgpfs/home/mkhairiansyah/.conda/envs/env-ml/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:597: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/mgpfs/home/mkhairiansyah/.conda/envs/env-ml/lib/python3.10/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


## Cell 4 — PES Analysis, Plotting, and Saving

Assembles the scan results into a summary DataFrame, plots the PES curve
(VQE vs FCI reference, and optionally an ORCA reference), and saves to CSV.

To overlay external ORCA energies, populate `orca_r` and `orca_e` lists
before calling `plot_pes()`.


In [ ]:
%matplotlib inline

In [ ]:
# ============================================================
# CELL 4 — PES ANALYSIS, PLOTTING, AND SAVING
# ============================================================

import numpy as np
from analysis     import create_pes_summary_table
from plotting     import plot_pes
from save_results import save_results
from IPython.display import display

# ── Sort results by bond length (in case of non-sequential runs) ──
pes_results_sorted = sorted(pes_results, key=lambda p: p["r"])

r_arr   = np.array([p["r"]          for p in pes_results_sorted])
e_vqe   = np.array([p["vqe_energy"] for p in pes_results_sorted])
e_fci   = np.array([p["fci_energy"] for p in pes_results_sorted])
err_mha = np.array([p["error_mha"]  for p in pes_results_sorted])

# ── Summary table ─────────────────────────────────────────
pes_df = create_pes_summary_table(EXPERIMENT, pes_results_sorted)
print("PES Summary Table:")
display(pes_df)

# ── Key statistics ────────────────────────────────────────
print()
print("=" * 58)
print("  PES STATISTICS")
print("=" * 58)
r_min_e  = r_arr[np.argmin(e_vqe)]
r_min_fci= r_arr[np.argmin(e_fci)]
print(f"  VQE minimum at      : {r_min_e:.4f} Å  ({np.min(e_vqe):.8f} Ha)")
print(f"  FCI minimum at      : {r_min_fci:.4f} Å  ({np.min(e_fci):.8f} Ha)")
print(f"  Max error (mHa)     : {err_mha.max():.4f}")
print(f"  Mean error (mHa)    : {err_mha.mean():.4f}")
_frac_chem = (err_mha < 1.0).sum()
print(f"  Chemical accuracy   : {_frac_chem}/{len(err_mha)} points")
print("=" * 58)

# ── PES plot ─────────────────────────────────────────────
# Optional: populate orca_r and orca_e to overlay ORCA reference.
# Example:
#   orca_r = [0.80, 0.90, ..., 2.00]   # Å
#   orca_e = [-40.123, -40.245, ...]    # Ha (CASSCF/CCSD(T)/etc.)
orca_r = None   # ← set to list[float] to enable ORCA overlay
orca_e = None   # ← set to list[float] to enable ORCA overlay

plot_pes(
    r_values=r_arr.tolist(),
    vqe_energies=e_vqe.tolist(),
    fci_energies=e_fci.tolist(),
    orca_energies=orca_e,
    orca_r_values=orca_r,
    molecule=EXPERIMENT["molecule"],
    scan_type=EXPERIMENT["scan_type"],
    title=(
        f"PES Scan — {EXPERIMENT['molecule']} / "
        f"{EXPERIMENT['ansatz'].upper()} / "
        f"{EXPERIMENT['basis']}  "
        f"[{EXPERIMENT['scan_type']}]"
    ),
    save_path=str(
        RESULTS_DIR / f"pes_{EXPERIMENT['molecule']}_{EXPERIMENT['basis']}_"
        f"{EXPERIMENT['ansatz']}_{EXPERIMENT['scan_type']}.png"
    ),
)

# ── Save CSV ─────────────────────────────────────────────
_fname = (
    f"pes_{EXPERIMENT['molecule']}_"
    f"{EXPERIMENT['basis']}_"
    f"{EXPERIMENT['ansatz']}_"
    f"{EXPERIMENT['scan_type']}.csv"
)
saved_to = save_results(str(RESULTS_DIR / _fname), pes_df, mode="w")
print(f"\nPES results saved to : {saved_to}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Data ORCA dari Benchmark_H2O_Klasik.ipynb ────────────
orca_r = np.linspace(0.20, 3.00, 16)

orca_hf = np.array([
    -65.806845162825, -72.793449568902, -74.429527148394,
    -74.867934490072, -74.962807989850, -74.945135249045,
    -74.889658857364, -74.824727459055, -74.761075397650,
    -74.703573381169, -74.654598768980, -74.614580215022,
    -74.582696023498, -74.557659137104, -74.538167465075,
    -74.523045762054
])
orca_casscf = np.array([
    -65.826140591478, -72.819970177160, -74.462339455416,
    -74.907776452539, -75.012412447078, -75.008837892233,
    -74.973175372795, -74.934507373189, -74.903075172534,
    -74.881421356997, -74.868316593036, -74.861071670803,
    -74.857277433606, -74.855356607624, -74.854411040105,
    -74.853958321358
])
orca_fci = np.array([
    -65.826174251028, -72.820020949044, -74.462413405084,
    -74.907839102179, -75.012460920566, -75.008875736423,
    -74.973205354435, -74.934531424790, -74.903094801021,
    -74.881437913534, -74.868331220728, -74.861085213714,
    -74.857290396455, -74.855369283774, -74.854423580792,
    -74.853970800445
])

# ── Plot terpadu: VQE + ORCA classical ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kiri: PES absolut semua metode
ax1 = axes[0]
ax1.plot(orca_r, orca_hf,     's:',  color='steelblue', lw=1.4, ms=5,
         label='HF/STO-3G (ORCA)')
ax1.plot(orca_r, orca_casscf, '^--', color='green',     lw=1.4, ms=5,
         label='CASSCF(8,6)/STO-3G (ORCA)')
ax1.plot(orca_r, orca_fci,    'x-',  color='crimson',   lw=1.4, ms=6,
         label='FCI/STO-3G (ORCA)')
ax1.plot(r_arr,  e_vqe,       'o-',  color='#2b6cb0',   lw=1.8, ms=5,
         label='VQE k-UpCCGSD/STO-3G')
ax1.axvline(0.9584, color='orange', ls='--', lw=1.2, alpha=0.7,
            label='R_exp = 0.9584 Å')
ax1.set_xlabel('O-H bond length (Å)', fontsize=11)
ax1.set_ylabel('Energy (Ha)', fontsize=11)
ax1.set_title('PES — H₂O: VQE vs Classical Methods', fontsize=11)
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.25, ls=':')

# Kanan: Error vs FCI untuk semua metode (region overlap VQE)
ax2 = axes[1]
# Interpolasi ORCA ke grid VQE untuk perbandingan fair
from scipy.interpolate import interp1d
_f_casscf = interp1d(orca_r, orca_casscf, kind='cubic')
_f_hf     = interp1d(orca_r, orca_hf,     kind='cubic')
_f_fci    = interp1d(orca_r, orca_fci,    kind='cubic')

_mask = (r_arr >= orca_r.min()) & (r_arr <= orca_r.max())
r_cmp = r_arr[_mask]

err_vqe    = np.abs(e_vqe[_mask]       - _f_fci(r_cmp)) * 1000
err_casscf = np.abs(_f_casscf(r_cmp)   - _f_fci(r_cmp)) * 1000
err_hf     = np.abs(_f_hf(r_cmp)       - _f_fci(r_cmp)) * 1000

ax2.semilogy(r_cmp, err_hf     + 1e-10, 's:', color='steelblue', lw=1.4, ms=5,
             label='|HF − FCI|')
ax2.semilogy(r_cmp, err_casscf + 1e-10, '^--', color='green', lw=1.4, ms=5,
             label='|CASSCF − FCI|')
ax2.semilogy(r_cmp, err_vqe    + 1e-10, 'o-',  color='#c05621', lw=1.8, ms=5,
             label='|VQE − FCI|')
ax2.axhline(1.0,  color='black', ls='--', lw=1.2, label='Chem. accuracy (1 mHa)')
ax2.axhline(1.6,  color='gray',  ls=':',  lw=1.0, label='1 kcal/mol (1.6 mHa)')
ax2.set_xlabel('O-H bond length (Å)', fontsize=11)
ax2.set_ylabel('|ΔE vs FCI| (mHa, log scale)', fontsize=11)
ax2.set_title('Energy error vs FCI — Overlap region (0.8–2.0 Å)', fontsize=11)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.25, ls=':', which='both')

fig.suptitle(
    'H₂O PES Benchmark: VQE k-UpCCGSD vs HF / CASSCF(8,6) / FCI  [STO-3G]',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'PES_H2O_unified_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CIRCUIT VISUALIZATION
# ============================================================

from qiskit_nature.second_q.drivers      import PySCFDriver
from qiskit_nature.units                 import DistanceUnit
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer
from qiskit.quantum_info                 import SparsePauliOp
from mapper_factory  import get_mapper
from ansatz_factory  import get_ansatz

# ── Bangun ulang ansatz di geometri equilibrium ───────────
_r_eq   = R_EQ.get(EXPERIMENT["molecule"].upper(), 1.0)
_atom   = build_geometry(EXPERIMENT["molecule"], _r_eq, EXPERIMENT["scan_type"])
_spin   = EXPERIMENT["multiplicity"] - 1

_driver = PySCFDriver(
    atom=_atom, unit=DistanceUnit.ANGSTROM,
    charge=EXPERIMENT["charge"], spin=_spin,
    basis=EXPERIMENT["basis"],
)
_prob = _driver.run()

if EXPERIMENT["freeze_core"]:
    _prob = FreezeCoreTransformer().transform(_prob)

if EXPERIMENT["use_active_space"]:
    _prob = ActiveSpaceTransformer(
        num_electrons=EXPERIMENT["active_electrons"],
        num_spatial_orbitals=EXPERIMENT["active_orbitals"],
    ).transform(_prob)

_use_2qr = (EXPERIMENT["mapper"] == "parity")
_mapper  = get_mapper(
    name=EXPERIMENT["mapper"],
    num_particles=_prob.num_particles if _use_2qr else None,
)

_ansatz_vis = get_ansatz(
    name=EXPERIMENT["ansatz"], problem=_prob,
    mapper=_mapper, reps=EXPERIMENT["reps"], k=EXPERIMENT["k"],
)

# ── Info dasar ────────────────────────────────────────────
print("=" * 52)
print(f"  CIRCUIT INFO — {EXPERIMENT['ansatz'].upper()}")
print("=" * 52)
print(f"  Molecule       : {EXPERIMENT['molecule']}")
print(f"  Basis          : {EXPERIMENT['basis']}")
print(f"  Num qubits     : {_ansatz_vis.num_qubits}")
print(f"  Num parameters : {_ansatz_vis.num_parameters}")
try:
    _d = _ansatz_vis.decompose()
    print(f"  Circuit depth  : {_d.depth()}")
    _ops = dict(_d.count_ops())
    print(f"  Gate counts    :")
    for _g, _c in sorted(_ops.items(), key=lambda x: -x[1]):
        print(f"    {_g:>12s} : {_c}")
except Exception:
    pass
print("=" * 52)

# ── 1. Full circuit (abstract) ────────────────────────────
print("\n--- Full Circuit (abstract) ---")
display(_ansatz_vis.draw('mpl', fold=-1, style='iqp'))

# ── 2. Decomposed (basis gates) ───────────────────────────
print("\n--- Decomposed Circuit (basis gates) ---")
try:
    from qiskit.compiler import transpile
    _transpiled = transpile(
        _ansatz_vis,
        basis_gates=['cx', 'u', 'rz', 'sx', 'x'],
        optimization_level=1,
    )
    print(f"  Depth after transpile : {_transpiled.depth()}")
    print(f"  CX count              : {dict(_transpiled.count_ops()).get('cx', 0)}")
    display(_transpiled.draw('mpl', fold=40, style='iqp'))
except Exception as e:
    print(f"Transpile skipped: {e}")

# ── 3. Parameter table ────────────────────────────────────
print("\n--- Parameters ---")
import pandas as pd
_param_df = pd.DataFrame({
    "Index": range(_ansatz_vis.num_parameters),
    "Name":  [str(p) for p in _ansatz_vis.parameters],
})
display(_param_df)